# Setup — add the "initiatives" table + 2 fields on projects

**This notebook WRITES to AGOL.** Run it once. You must be the **owner or an admin** of the `datateam_portfolio_v2` feature service.

Two changes:
1. Adds a new `initiatives` table in `datateam_portfolio_v2` (next layer index, alongside projects / tasks / notes / reviews / status history / notifications).
2. Adds two new fields to the existing `projects` layer: `initiative_id` (FK to the new table) and `initiative_sequence` (manual ordering within an initiative).

After it runs, copy the printed table URL into `ARCGIS_CONFIG.initiativesUrl` in `src/agol.js`.

**Schema rationale**
- One row per initiative — first-class entity with its own status / owner / dates.
- Projects gain a single FK (one initiative per project) plus a sequence integer for the drag-to-reorder UX inside the initiative detail page.
- No dependency edges in V1 — sequence is visual ordering only.

In [ ]:
from arcgis.gis import GIS
from arcgis.features import FeatureLayerCollection

gis = GIS("home")
print(f"Signed in as {gis.users.me.username} @ {gis.url}")

SERVICE_URL = "https://services3.arcgis.com/9coHY2fvuFjG9HQX/ArcGIS/rest/services/datateam_portfolio_v2/FeatureServer"
flc = FeatureLayerCollection(SERVICE_URL, gis)

existing_layers = [l.properties.name for l in flc.layers]
existing_tables = [t.properties.name for t in flc.tables]
print("Layers:", existing_layers)
print("Tables:", existing_tables)

## 1. Initiatives table definition

Fields beyond OBJECTID:
- `initiative_id` (Text 50) — short slug for the human-readable ID; app generates one on create.
- `name` (Text 255) — display title.
- `description` (Text 4000) — leadership summary; renders as the hero sub-headline.
- `owner` (Text 255) — single accountable person.
- `status` (Text 50) — Planning / Active / On Hold / Complete / Canceled.
- `target_start` (Date Only) — when the initiative was supposed to begin.
- `target_completion` (Date Only) — the leadership-facing target date.
- `strategic_alignment` (Text 100, nullable) — optional tag pointing to Data Program / Safe City / etc.

In [ ]:
TABLE_NAME = "initiatives"

table_def = {
    "type": "Table",
    "name": TABLE_NAME,
    "description": "Strategic-objective parent of related projects. Each row owns a set of projects via projects.initiative_id.",
    "hasAttachments": False,
    "hasM": False,
    "hasZ": False,
    "objectIdField": "OBJECTID",
    "globalIdField": "",
    "supportsAdvancedQueries": True,
    "allowGeometryUpdates": False,
    "capabilities": "Create,Delete,Query,Update,Editing",
    "fields": [
        {"name": "OBJECTID",            "type": "esriFieldTypeOID",      "alias": "OBJECTID",            "nullable": False, "editable": False},
        {"name": "initiative_id",       "type": "esriFieldTypeString",   "alias": "Initiative ID",       "length": 50,   "nullable": False, "editable": True},
        {"name": "name",                "type": "esriFieldTypeString",   "alias": "Name",                "length": 255,  "nullable": False, "editable": True},
        {"name": "description",         "type": "esriFieldTypeString",   "alias": "Description",         "length": 4000, "nullable": True,  "editable": True},
        {"name": "owner",               "type": "esriFieldTypeString",   "alias": "Owner",               "length": 255,  "nullable": True,  "editable": True},
        {"name": "status",              "type": "esriFieldTypeString",   "alias": "Status",              "length": 50,   "nullable": True,  "editable": True, "defaultValue": "Planning"},
        {"name": "target_start",        "type": "esriFieldTypeDateOnly", "alias": "Target Start",        "nullable": True,  "editable": True},
        {"name": "target_completion",  "type": "esriFieldTypeDateOnly", "alias": "Target Completion",  "nullable": True,  "editable": True},
        {"name": "strategic_alignment","type": "esriFieldTypeString",   "alias": "Strategic Alignment", "length": 100,  "nullable": True,  "editable": True}
    ],
    "indexes": [
        {"name": "init_id_idx",     "fields": "initiative_id", "isAscending": True, "isUnique": True,  "description": "unique slug lookup"},
        {"name": "init_status_idx", "fields": "status",        "isAscending": True, "isUnique": False, "description": "status filter"}
    ]
}

print("Defined", len(table_def["fields"]), "fields for table:", TABLE_NAME)

In [ ]:
if TABLE_NAME in existing_tables:
    print(f"'{TABLE_NAME}' already exists in this service — skipping create.")
else:
    result = flc.manager.add_to_definition({"tables": [table_def]})
    print("add_to_definition result:", result)

## 2. Add `initiative_id` + `initiative_sequence` fields to the projects layer

Projects layer is index 0 in this FeatureServer. Both new fields are nullable — every existing project remains valid (no initiative, no sequence) until an admin attaches it.

In [ ]:
projects_layer = flc.layers[0]
existing_project_fields = {f["name"] for f in projects_layer.properties.fields}
print("Projects layer:", projects_layer.properties.name)
print("Has initiative_id?", "initiative_id" in existing_project_fields)
print("Has initiative_sequence?", "initiative_sequence" in existing_project_fields)

fields_to_add = []
if "initiative_id" not in existing_project_fields:
    fields_to_add.append({
        "name": "initiative_id",
        "type": "esriFieldTypeString",
        "alias": "Initiative ID",
        "length": 50,
        "nullable": True,
        "editable": True
    })
if "initiative_sequence" not in existing_project_fields:
    fields_to_add.append({
        "name": "initiative_sequence",
        "type": "esriFieldTypeInteger",
        "alias": "Initiative Sequence",
        "nullable": True,
        "editable": True
    })

if not fields_to_add:
    print("Both fields already present — skipping field add.")
else:
    result = projects_layer.manager.add_to_definition({"fields": fields_to_add})
    print("Field add result:", result)

## Verify + get the URLs

Re-reads the service and prints the new table's REST URL plus the projects layer's confirmed fields.

In [ ]:
flc2 = FeatureLayerCollection(SERVICE_URL, gis)
found = None
for t in flc2.tables:
    if t.properties.name == TABLE_NAME:
        found = t
        break

if not found:
    print("Could not find the new initiatives table — check the add_to_definition result above.")
else:
    print("initiativesUrl:", found.url)
    print()
    print("Fields:")
    for f in found.properties.fields:
        print(f"  {f['name']:22s} {f['type']}")

print()
print("Projects layer now has these fields (looking for initiative_id + initiative_sequence):")
for f in flc2.layers[0].properties.fields:
    if "initiative" in f["name"]:
        print(f"  {f['name']:22s} {f['type']}")

## Next step

In `src/agol.js`, add to `ARCGIS_CONFIG` (next to `notificationsUrl`):

    initiativesUrl:    '<paste the URL printed above>',

The app code is already feature-detected — it'll show an empty Initiatives list ("No initiatives yet — create one") until you add the URL, then the feature activates without another code change.

**To roll back later** (cleanup):

    # Remove the initiatives table
    flc.manager.delete_from_definition({"tables": [{"name": "initiatives"}]})
    # Remove the two FK fields from projects
    flc.layers[0].manager.delete_from_definition({"fields": [{"name": "initiative_id"}, {"name": "initiative_sequence"}]})

Or delete from the service's Data tab in AGOL.